# Distributed Batch Inference with Ray Core

This notebook performs **distributed batch inference** using **Ray Core** on models registered in Unity Catalog:
- Loads the latest versions of CPU models (`cpu_model_*_child`)
- Distributes inference tasks across Ray workers
- Writes predictions from all models to a single Delta table

**Key Feature**: Uses Ray Core for parallel model loading and inference across the cluster.

**Cluster Configuration**: CPU cluster (8 workers, 32 cores per node = 256 total cores)

**Note**: Run this notebook on a CPU cluster for inference.

## 1. Setup and Imports

In [ ]:
# Core libraries
import numpy as np
import pandas as pd
from datetime import datetime
import json
import time
import os
from typing import List, Dict, Any, Tuple

# Ray imports
import ray

# MLflow for model loading from Unity Catalog
import mlflow
from mlflow.tracking import MlflowClient

# PySpark
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField, IntegerType, DoubleType, 
    StringType, ArrayType, TimestampType
)

# Scikit-learn for preprocessing
from sklearn.preprocessing import StandardScaler

print("All imports successful!")
print(f"Notebook type: CPU CLUSTER - Distributed Batch Inference with Ray Core")

## 2. Configuration

In [ ]:
# Configuration parameters
catalog = "ryuta"
schema = "ray"
experiment_name = "/Users/ryuta.yoshimatsu@databricks.com/ray_batch_inference"

print(f"Running with parameters:")
print(f"  Catalog: {catalog}")
print(f"  Schema: {schema}")
print(f"  Experiment: {experiment_name}")

# Configuration
CONFIG = {
    'catalog': catalog,
    'schema': schema,
    'source_table': f'{catalog}.{schema}.synthetic_data',
    'inference_results_table': f'{catalog}.{schema}.batch_inference_results',
    
    # Model registry path (Unity Catalog)
    'model_registry_path': f'{catalog}.{schema}',
    
    # Model name pattern to search for (cpu_model_*_child only)
    'model_pattern': 'cpu_model_',
    'model_suffix': '_child',
    
    # Cluster configuration
    'cluster_type': 'cpu',
    'n_workers': 8,
    'cores_per_head_node': 0,
    'cores_per_node': 32,
    'total_cores': 256,
    
    # MLflow experiment for tracking
    'experiment_name': experiment_name
}

# Set up MLflow to use Unity Catalog
mlflow.set_registry_uri("databricks-uc")
mlflow.set_experiment(CONFIG['experiment_name'])

# Get Databricks credentials for Ray workers
# These will be passed to each Ray task to enable model loading from Unity Catalog
from mlflow.utils.databricks_utils import get_databricks_env_vars
mlflow_db_creds = get_databricks_env_vars("databricks")

print("\nConfiguration loaded successfully!")
print(json.dumps(CONFIG, indent=2))
print(f"\nMLflow registry URI: {mlflow.get_registry_uri()}")
print(f"Databricks credentials captured for Ray workers: {list(mlflow_db_creds.keys())}")

## 3. Discover Registered Models in Unity Catalog

In [ ]:
def get_registered_models(catalog: str, schema: str, model_pattern: str, model_suffix: str) -> List[Dict[str, Any]]:
    """
    Discover registered models in the specified Unity Catalog namespace matching the pattern.
    Returns a list of model info dictionaries with name and latest version.
    
    Note: Unity Catalog doesn't support filter_string in search_registered_models,
    so we list all models and filter locally.
    
    Args:
        catalog: Unity Catalog name
        schema: Schema name
        model_pattern: Model name prefix (e.g., 'cpu_model_')
        model_suffix: Model name suffix (e.g., '_child')
    """
    client = MlflowClient()
    
    # Search for models in the Unity Catalog namespace
    model_prefix = f"{catalog}.{schema}"
    
    models = []
    
    try:
        # For Unity Catalog, list all models with pagination handling
        # The API returns a PagedList, so we need to iterate through all pages
        all_models = []
        page_token = None
        
        while True:
            # Fetch a page of results
            result = client.search_registered_models(
                max_results=100,  # Page size
                page_token=page_token
            )
            all_models.extend(result)
            
            # Check if there are more pages
            page_token = getattr(result, 'token', None)
            if not page_token:
                break
        
        print(f"Total models fetched from registry: {len(all_models)}")
        
        for model in all_models:
            model_name = model.name
            
            # Check if model is in our catalog.schema namespace
            if not model_name.startswith(f"{model_prefix}."):
                continue
                
            short_name = model_name.replace(f"{model_prefix}.", "")
            
            # Verify it matches our exact pattern (cpu_model_*_child)
            if short_name.startswith(model_pattern) and short_name.endswith(model_suffix):
                # Get version info from the registered model
                # The latest_versions attribute contains version info
                if hasattr(model, 'latest_versions') and model.latest_versions:
                    latest_version = model.latest_versions[0]
                    version = latest_version.version
                    run_id = latest_version.run_id
                    status = latest_version.status
                    source = latest_version.source
                else:
                    # Search for versions if not available directly
                    versions = client.search_model_versions(f"name='{model_name}'")
                    if versions:
                        # Get the latest version (highest version number)
                        latest = max(versions, key=lambda v: int(v.version))
                        version = latest.version
                        run_id = latest.run_id
                        status = latest.status
                        source = latest.source
                    else:
                        continue
                
                models.append({
                    'model_name': model_name,
                    'short_name': short_name,
                    'version': version,
                    'model_type': 'cpu',
                    'run_id': run_id,
                    'status': status,
                    'source': source
                })
        
        print(f"Found {len(models)} registered models matching '{model_pattern}*{model_suffix}' in {model_prefix}")
        
    except Exception as e:
        print(f"Error searching for models: {e}")
        import traceback
        traceback.print_exc()
        raise
    
    return models

# Discover registered models (cpu_model_*_child only)
print(f"Searching for models in {CONFIG['catalog']}.{CONFIG['schema']}...")
print(f"Pattern: {CONFIG['model_pattern']}*{CONFIG['model_suffix']}")
registered_models = get_registered_models(
    CONFIG['catalog'], 
    CONFIG['schema'],
    CONFIG['model_pattern'],
    CONFIG['model_suffix']
)

# Display summary
print(f"\nModel Summary:")
print(f"  Total models: {len(registered_models)}")

# Show first few models
if registered_models:
    print(f"\nFirst 5 models:")
    for model in registered_models[:5]:
        print(f"  - {model['short_name']} (v{model['version']})")
else:
    print("\nNo models found matching the pattern.")

## 4. Load Data for Inference

In [ ]:
# Load data from Delta table
print(f"Loading data from {CONFIG['source_table']}...")
df_spark = spark.table(CONFIG['source_table'])

total_rows = df_spark.count()
print(f"Total rows: {total_rows}")

# Convert to pandas for inference
df = df_spark.toPandas()
print(f"Data loaded successfully! Shape: {df.shape}")

# Prepare features
feature_columns = [col for col in df.columns if col.startswith('feature_')]
X = df[feature_columns].values

# Get actual labels if available (for evaluation)
if 'label' in df.columns:
    y_actual = df['label'].values
    print(f"Actual labels available for evaluation")
else:
    y_actual = None
    print(f"No labels available - inference only")

print(f"\nFeatures shape: {X.shape}")
print(f"Feature columns: {len(feature_columns)}")

## 5. Initialize Ray Cluster

In [ ]:
# Initialize Ray cluster using Databricks utilities
from ray.util.spark import setup_ray_cluster, shutdown_ray_cluster

# Shutdown any existing Ray instance
if ray.is_initialized():
    ray.shutdown()

# Check the cluster configuration first
print("Checking Spark cluster configuration...")
sc = spark.sparkContext
num_executors = sc._jsc.sc().getExecutorMemoryStatus().size() - 1  # Subtract 1 for driver
print(f"Number of Spark executors: {num_executors}")

# Adjust Ray config based on actual cluster
if num_executors > 0:
    n_workers = num_executors
else:
    n_workers = 1  # Single node mode

print(f"Setting up Ray with {n_workers} workers...")

try:
    # Setup Ray cluster spanning all Spark nodes
    setup_ray_cluster(
        min_worker_nodes=n_workers,
        max_worker_nodes=n_workers,
        num_cpus_head_node=0,
        num_cpus_worker_node=CONFIG['cores_per_node'],
        num_gpus_head_node=0,
        num_gpus_worker_node=0,
        collect_log_to_path="/Workspace/Users/ryuta.yoshimatsu@databricks.com/ray_logs",
    )
    
    # Initialize Ray client
    ray.init(
        address='auto',
        ignore_reinit_error=True,
        logging_level='ERROR',
    )
    
except Exception as e:
    print(f"Warning: setup_ray_cluster failed with: {e}")
    print("Falling back to standard Ray initialization...")
    ray.init(
        ignore_reinit_error=True,
        logging_level='ERROR',
    )

print("\nRay initialized successfully!")
print(f"Available CPUs: {ray.cluster_resources().get('CPU', 0)}")
print(f"Available GPUs: {ray.cluster_resources().get('GPU', 0)}")
print(f"Cluster nodes connected: {len(ray.nodes())}")
print(f"Total cluster resources: {ray.cluster_resources()}")

## 6. Ray Remote Function for Model Inference

Define a Ray remote function that:
1. Loads a model from Unity Catalog
2. Performs inference on the input data
3. Returns predictions

In [ ]:
@ray.remote
def run_model_inference(
    model_info: Dict[str, Any],
    X_data: np.ndarray,
    row_indices: np.ndarray,
    mlflow_db_creds: Dict[str, str]
) -> Dict[str, Any]:
    """
    Ray remote function to load a model from Unity Catalog and perform inference.
    Accepts Databricks credentials to authenticate with Unity Catalog.
    
    Args:
        model_info: Dictionary containing model metadata (name, version, type)
        X_data: Feature matrix for inference
        row_indices: Original row indices for tracking
        mlflow_db_creds: Databricks credentials for Unity Catalog access
        
    Returns:
        Dictionary containing model name, predictions, and probabilities
    """
    import os
    import time
    import numpy as np
    import mlflow
    from sklearn.preprocessing import StandardScaler
    
    start_time = time.time()
    model_name = model_info['model_name']
    model_version = model_info['version']
    
    try:
        # Set Databricks credentials in the worker environment
        os.environ.update(mlflow_db_creds)
        
        # Set MLflow registry to Unity Catalog
        mlflow.set_registry_uri("databricks-uc")
        
        # Load the model from Unity Catalog
        model_uri = f"models:/{model_name}/{model_version}"
        
        # Load model using pyfunc (works for all model types)
        model = mlflow.pyfunc.load_model(model_uri)
        
        # Load feature indices artifact if available
        run_id = model_info.get('run_id')
        feature_indices = None
        
        if run_id:
            try:
                import json
                artifact_path = mlflow.artifacts.download_artifacts(
                    run_id=run_id,
                    artifact_path="feature_indices.json"
                )
                with open(artifact_path, 'r') as f:
                    feature_indices = json.load(f)
            except Exception:
                # Feature indices not available, use all features
                feature_indices = None
        
        # Apply feature selection if indices are available
        if feature_indices is not None:
            X_subset = X_data[:, feature_indices]
        else:
            X_subset = X_data
        
        # Scale features (models were trained with scaled data)
        scaler = StandardScaler()
        X_scaled = scaler.fit_transform(X_subset)
        
        # Perform inference
        predictions = model.predict(X_scaled)
        
        # Handle different output formats
        if isinstance(predictions, np.ndarray):
            if predictions.ndim == 2:
                if predictions.shape[1] == 1:
                    probabilities = predictions.flatten()
                else:
                    probabilities = predictions[:, 1] if predictions.shape[1] == 2 else predictions[:, 0]
            else:
                probabilities = predictions.flatten()
        else:
            probabilities = np.array(predictions).flatten()
        
        # Convert probabilities to binary predictions
        binary_predictions = (probabilities > 0.5).astype(int)
        
        inference_time = time.time() - start_time
        
        return {
            'model_name': model_name,
            'model_short_name': model_info['short_name'],
            'model_version': model_version,
            'model_type': model_info['model_type'],
            'row_indices': row_indices.tolist(),
            'probabilities': probabilities.tolist(),
            'predictions': binary_predictions.tolist(),
            'n_samples': len(predictions),
            'inference_time': inference_time,
            'status': 'success'
        }
        
    except Exception as e:
        import traceback
        return {
            'model_name': model_name,
            'model_short_name': model_info.get('short_name', 'unknown'),
            'model_version': model_version,
            'model_type': model_info.get('model_type', 'unknown'),
            'row_indices': row_indices.tolist() if row_indices is not None else [],
            'probabilities': [],
            'predictions': [],
            'n_samples': 0,
            'inference_time': time.time() - start_time,
            'status': 'failed',
            'error': str(e),
            'traceback': traceback.format_exc()
        }

print("Ray remote inference function defined!")

## 7. Run Distributed Batch Inference

In [ ]:
# Put data in Ray object store
row_indices = np.arange(len(X))
X_ref = ray.put(X)
row_indices_ref = ray.put(row_indices)

print(f"Data stored in Ray object store")
print(f"  Samples: {len(X)}")
print(f"  Features: {X.shape[1]}")

In [ ]:
# Launch distributed inference across all models
print(f"\nLaunching distributed inference for {len(registered_models)} models...")
print("="*80)
start_time = time.time()

# Submit inference tasks for all models (pass credentials for Unity Catalog access)
futures = []
for model_info in registered_models:
    future = run_model_inference.remote(
        model_info=model_info,
        X_data=X_ref,
        row_indices=row_indices_ref,
        mlflow_db_creds=mlflow_db_creds
    )
    futures.append(future)

print(f"Submitted {len(futures)} inference tasks")
print(f"\nCollecting results...")

# Collect results as they complete
inference_results = []
completed = 0
remaining_futures = futures.copy()

while remaining_futures:
    ready_futures, remaining_futures = ray.wait(remaining_futures, num_returns=1)
    
    for future in ready_futures:
        result = ray.get(future)
        inference_results.append(result)
        completed += 1
        
        if result['status'] == 'success':
            print(f"[{completed}/{len(futures)}] {result['model_short_name']} - "
                  f"{result['n_samples']} samples ({result['inference_time']:.2f}s)")
        else:
            print(f"[{completed}/{len(futures)}] {result['model_short_name']} - FAILED: {result.get('error', 'Unknown')}")

total_inference_time = time.time() - start_time

# Summary
successful_results = [r for r in inference_results if r['status'] == 'success']
failed_results = [r for r in inference_results if r['status'] != 'success']

print(f"\n{'='*80}")
print(f"INFERENCE COMPLETE")
print(f"{'='*80}")
print(f"Total time: {total_inference_time:.2f}s")
print(f"Successful: {len(successful_results)}")
print(f"Failed: {len(failed_results)}")

## 7.1 Shutdown Ray Cluster

Shutdown Ray cluster immediately after inference is complete to free up resources before writing to Delta.

In [ ]:
# Shutdown Ray cluster after inference is complete
# This frees up resources before Delta table operations
from ray.util.spark import shutdown_ray_cluster

print("\nShutting down Ray cluster...")
try:
    shutdown_ray_cluster()
    print("Ray cluster shut down successfully!")
except Exception as e:
    print(f"Warning: Ray cluster shutdown encountered an error (non-critical): {e}")

try:
    ray.shutdown()
    print("Ray client shut down successfully!")
except Exception as e:
    print(f"Warning: Ray client shutdown encountered an error: {e}")

print("\nRay resources released. Proceeding with Delta table operations...")

## 8. Prepare Inference Results for Delta Table

In [ ]:
# Prepare inference results from all models for Delta table
print("\nPreparing inference results from all models...")

inference_timestamp = datetime.now()
n_successful_models = len(successful_results)

# Create DataFrame with all model predictions
inference_records = []

for result in successful_results:
    model_name = result['model_short_name']
    model_type = result['model_type']
    model_version = result['model_version']
    
    for row_idx, prob, pred in zip(
        result['row_indices'],
        result['probabilities'],
        result['predictions']
    ):
        record = {
            'row_index': int(row_idx),
            'model_name': model_name,
            'model_version': str(model_version),
            'probability': float(prob),
            'prediction': int(pred),
            'inference_timestamp': inference_timestamp
        }
        
        # Add actual label if available
        if y_actual is not None:
            record['actual_label'] = int(y_actual[row_idx])
            record['is_correct'] = int(pred == y_actual[row_idx])
        
        inference_records.append(record)

# Create DataFrame
inference_df = pd.DataFrame(inference_records)

print(f"Inference results prepared from {n_successful_models} models")
print(f"Total prediction records: {len(inference_records):,}")
print(f"DataFrame shape: {inference_df.shape}")

# Show sample
print("\nSample results:")
inference_df.head(10)

## 9. Write Results to Delta Table

In [ ]:
# Convert to Spark DataFrame and write to Delta table
print(f"\nWriting inference results to {CONFIG['inference_results_table']}...")

inference_spark_df = spark.createDataFrame(inference_df)

# Write to Delta table (overwrite mode)
inference_spark_df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(CONFIG['inference_results_table'])

print(f"Inference results written successfully!")
print(f"  Table: {CONFIG['inference_results_table']}")
print(f"  Rows: {inference_spark_df.count():,}")

In [ ]:
# Verify the written results
print(f"\nVerifying {CONFIG['inference_results_table']}...")
verify_df = spark.table(CONFIG['inference_results_table'])

print(f"\nSchema:")
verify_df.printSchema()

print(f"\nSample data:")
verify_df.show(10, truncate=False)

## 10. Analyze Results by Model

In [ ]:
# Analyze predictions by model
print("\nPredictions summary by model:")
verify_df.groupBy('model_name') \
    .agg(
        F.count('*').alias('n_predictions'),
        F.avg('probability').alias('avg_probability'),
        F.sum('prediction').alias('positive_predictions')
    ) \
    .orderBy('model_name') \
    .show(100, truncate=False)

In [ ]:
# If labels are available, show accuracy by model
if 'is_correct' in verify_df.columns:
    print("\nAccuracy by model:")
    verify_df.groupBy('model_name') \
        .agg(
            F.avg('is_correct').alias('accuracy'),
            F.count('*').alias('n_predictions')
        ) \
        .orderBy(F.desc('accuracy')) \
        .show(100, truncate=False)
    
    # Overall accuracy across all models
    overall_accuracy = verify_df.select(F.avg('is_correct')).collect()[0][0]
    print(f"\nOverall accuracy (all models combined): {overall_accuracy:.4f}")

## 11. Log Inference Run to MLflow

In [ ]:
# Log inference run to MLflow for tracking
print("\nLogging inference run to MLflow...")

n_samples = len(X)

with mlflow.start_run(run_name="batch_inference_run") as run:
    # Log parameters
    mlflow.log_params({
        'n_samples': n_samples,
        'n_features': X.shape[1],
        'n_models_attempted': len(registered_models),
        'n_models_successful': len(successful_results),
        'n_models_failed': len(failed_results),
        'source_table': CONFIG['source_table'],
        'results_table': CONFIG['inference_results_table']
    })
    
    # Log metrics
    mlflow.log_metrics({
        'total_inference_time': total_inference_time,
        'avg_model_inference_time': np.mean([r['inference_time'] for r in successful_results]),
        'total_predictions': len(inference_records)
    })
    
    # Log list of models used
    model_list = [r['model_short_name'] for r in successful_results]
    mlflow.log_text(json.dumps(model_list, indent=2), "models_used.json")
    
    # Log failed models if any
    if failed_results:
        failed_info = [{
            'model': r['model_short_name'],
            'error': r.get('error', 'Unknown')
        } for r in failed_results]
        mlflow.log_text(json.dumps(failed_info, indent=2), "failed_models.json")
    
    print(f"MLflow run ID: {run.info.run_id}")

print("\nInference run logged to MLflow successfully!")

In [ ]:
# This cell intentionally left empty - MLflow logging completed above

## 12. Ray Cluster Status

**Note:** Ray cluster was already shut down in Section 7.1 (immediately after inference completed) to free up resources for Delta table operations.

In [ ]:
# Verify Ray cluster status
print("Ray cluster status check:")
if ray.is_initialized():
    print("  Warning: Ray is still initialized (unexpected)")
else:
    print("  Ray cluster was successfully shut down in Section 7.1")

## Summary

In [ ]:
# Summary statistics
print("="*80)
print("BATCH INFERENCE SUMMARY")
print("="*80)
print(f"Source table: {CONFIG['source_table']}")
print(f"Results table: {CONFIG['inference_results_table']}")
print(f"Models used: {len(successful_results)}")
print(f"Total predictions: {len(inference_records):,}")
print(f"Total inference time: {total_inference_time:.2f}s")
print("="*80)

This notebook performed **distributed batch inference** using **Ray Core**:

**What was done:**
1. Discovered all registered `cpu_model_*_child` models in Unity Catalog
2. Loaded the latest version of each model
3. Distributed inference tasks across Ray workers
4. Wrote all model predictions to a single Delta table

**Output Table:**
- `{catalog}.{schema}.batch_inference_results` - Predictions from all models

**Table Schema:**
- `row_index` - Index of the input row
- `model_name` - Name of the model that made the prediction
- `model_version` - Version of the model
- `probability` - Predicted probability
- `prediction` - Binary prediction (0 or 1)
- `actual_label` - Actual label (if available)
- `is_correct` - Whether prediction was correct (if labels available)
- `inference_timestamp` - When inference was performed

**Key Features:**
- Ray Core for distributed model loading and inference
- Parallel inference across all available CPU cores
- All model predictions in a single table for easy analysis
- MLflow tracking for inference runs

**Next Steps:**
1. Query the results table to compare model predictions
2. Analyze which models perform best on different subsets
3. Use the predictions for downstream applications